# WTI Flat Price Contracts — Open Interest EDA

Analyze the relative size of each WTI flat price contract code from `disaggregated_combined.csv`.

Flat price codes (from `docs/wti_cot_mapping.md`):
- `067651` — CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE (physical WTI)
- `067411` — CRUDE OIL, LIGHT SWEET - ICE FUTURES EUROPE (CFTC-reported)
- `06765I` — WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `067A55` — Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `067655` — E-MINI CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE
- `06765C` — CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06765B` — EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06741Q` — WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV
- `06765A` — WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANTILE EXCHANGE
- `06765G` — DUBAI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANTILE EXCHANGE

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)
print(f"Total rows: {len(df):,}")
df.head(2)

In [ ]:
WTI_FLAT_PRICE_CODES = [
    "067651",  # CRUDE OIL, LIGHT SWEET - NYMEX (physical)
    "067411",  # CRUDE OIL, LIGHT SWEET - ICE FUTURES EUROPE
    "06765I",  # WTI CRUDE OIL FINANCIAL - NYMEX
    "067A55",  # Micro WTI CRUDE OIL FINANCIAL - NYMEX
    "067655",  # E-MINI CRUDE OIL, LIGHT SWEET - NYMEX
    "06765C",  # CRUDE OIL AVG PRICE OPTIONS - NYMEX
    "06765B",  # EUR STYLE CRUDE OIL OPTIONS - NYMEX
    "06741Q",  # WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV
    "06765A",  # WTI CRUDE OIL CALENDAR SWAP - NYMEX (added per Marouen)
    "06765G",  # DUBAI CRUDE OIL CALENDAR SWAP - NYMEX (added per Marouen)
]

wti = df[df["cftc_contract_market_code"].isin(WTI_FLAT_PRICE_CODES)].copy()
wti["report_date_as_yyyy_mm_dd"] = pd.to_datetime(wti["report_date_as_yyyy_mm_dd"])
wti["open_interest_all"] = pd.to_numeric(wti["open_interest_all"], errors="coerce")
print(f"WTI flat price rows: {len(wti):,}")
print(f"Date range: {wti['report_date_as_yyyy_mm_dd'].min().date()} to {wti['report_date_as_yyyy_mm_dd'].max().date()}")
print(f"Codes found: {sorted(wti['cftc_contract_market_code'].unique())}")

## Average Open Interest by Contract Code

In [ ]:
# Build a readable label for each code
code_labels = (
    wti.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

avg_oi = (
    wti.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = "CFTC"
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

## Open Interest Share — Pie Chart

In [ ]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="WTI Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract Code

In [ ]:
fig = px.line(
    wti.sort_values("report_date_as_yyyy_mm_dd"),
    x="report_date_as_yyyy_mm_dd",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="WTI Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date_as_yyyy_mm_dd": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [ ]:
# Pivot to get OI per code per date, then compute percentage share
pivot = wti.pivot_table(
    index="report_date_as_yyyy_mm_dd",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="WTI Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total WTI Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [ ]:
latest_date = wti["report_date_as_yyyy_mm_dd"].max()
latest = wti[wti["report_date_as_yyyy_mm_dd"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary